<a href="https://colab.research.google.com/github/pratbharat/-advancecfdwithpratyush/blob/main/2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Simple Neural Network
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
#Create Neural Network Model = d2T/dx2 = 0
class PINN_Model(nn.Module):
   #Input Layer : 2 -> Output Layer -> 1
   #Hidden -> 3 , 20-> 20-> 20
    def __init__(self,in_feat= 2, h1=20, h2= 20, h3=20, out=1):
        super(PINN_Model, self).__init__()
        self.fc1 = nn.Linear(in_feat,h1)
        self.fc2 = nn.Linear(h1,h2)
        self.fc3 = nn.Linear(h2,h3)
        self.out = nn.Linear(h3,out)
   #RELU= <0 0> linear =>
   #Tanh = Continuous

    def forward(self,x):
        self.activation = nn.Tanh()
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.out(x)
        return x

In [4]:
model = PINN_Model()
print(model)

PINN_Model(
  (fc1): Linear(in_features=2, out_features=20, bias=True)
  (fc2): Linear(in_features=20, out_features=20, bias=True)
  (fc3): Linear(in_features=20, out_features=20, bias=True)
  (out): Linear(in_features=20, out_features=1, bias=True)
)


In [ ]:
#Loss  = pde_loss + loss_BC + loss_IC
def pde_residual(model,x1, x2):
  #Function = T''(x1) + T''(x2) = 0
  x1 = x1.clone().detach().requires_grad_(True)
  x2 = x2.clone().detach().requires_grad_(True)

  T = model(x1,x2)

  #Compute the first order derivative
  T_x = torch.autograd.grad(T,x,grad_outputs=torch.ones_like(T),create_graph=True)[0]

  #Second order derivative
  T_xx = torch.autograd.grad(T_x,x,grad_outputs=torch.ones_like(T_x),create_graph=True)[0]

  #Residual
  residual = T_xx + torch.pi**2 * T
  return residual